# Single-task augmentation generalization continuation on Modal

This warm-start experiment continues the verified step-512 transformer from run `20260826T155954304798Z` through global step 2048. It rebuilds and hash-checks the identical 512-view training pool and 128-view holdout, then adds three more 512-step sampling phases. NVARC and the FLUX VAE stay frozen; the full FLUX transformer remains trainable.

The parent checkpoint contains transformer weights only, so AdamW starts fresh. This is a documented warm start, not an exact optimizer-state resume.

## 1. Immutable runtime configuration

Set `DIFFUSION_ARC_REVISION` to the exact pushed 40-character commit containing this notebook and runner. The parent report ID and SHA-256 are pinned below. Automatic shutdown is enabled unless `DIFFUSION_ARC_AUTO_SHUTDOWN=0` is set.

In [ ]:
from __future__ import annotations

import base64
from collections import deque
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys
import time

VOLUME_NAME = 'diffusion-arc-models'
VOLUME_ROOT = Path('/mnt/diffusion-arc-models')
REPOSITORY_URL = 'https://github.com/joshuakauling/diffusion-arc.git'
SOURCE_REVISION = "ae1bc579844a67ea214f9389447fca7c6f6efbc1"
AUTO_SHUTDOWN = os.environ.get('DIFFUSION_ARC_AUTO_SHUTDOWN', '1') == '1'

PROTOCOL = 'augmentation-generalization-overfit-continuation'
RESULT_FAMILY = 'single-task-augmentation-generalization-overfitting'
EXPECTED_STAGE = 'augmentation-generalization-overfit'
EXPECTED_INITIAL_STEP = 512
EXPECTED_FINAL_STEP = 2048
EXPECTED_EVALUATION_STEPS = [1024, 2048]
EXPECTED_EVALUATION_KINDS = ['t1-only', 'full-generation']
PARENT_RUN_ID = '20260826T155954304798Z'
PARENT_REPORT_SHA256 = '0480c43fcf7349614a79a4da33f780dd930484cbae7cf06c8b24248d35c7d5cf'
PARENT_REPORT_PATH = (
    VOLUME_ROOT / 'results' / RESULT_FAMILY / PARENT_RUN_ID / 'report.json'
)
PREVIEW_PAGE_LIMIT = 12

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE_DIR = Path('/tmp') / f'diffusion-arc-augmentation-continuation-source-{RUN_ID}'
REPORT_PATH = VOLUME_ROOT / 'results' / RESULT_FAMILY / RUN_ID / 'report.json'

PINNED_PACKAGES = {
    'accelerate': '1.12.0',
    'diffusers': '0.39.0',
    'einops': '0.8.1',
    'Pillow': '11.3.0',
    'safetensors': '0.8.0',
    'torch': '2.13.0',
    'tqdm': '4.67.1',
    'transformers': '4.56.1',
}


def require_exact_commit(revision: str) -> None:
    if len(revision) != 40 or any(
        character not in '0123456789abcdefABCDEF' for character in revision
    ):
        raise ValueError(
            'DIFFUSION_ARC_REVISION must be an exact pushed 40-character commit'
        )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def run_command(
    label: str,
    command: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
) -> str:
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        tail = '\n'.join(completed.stdout.splitlines()[-50:])
        if tail:
            print(tail)
        raise RuntimeError(f'{label} failed with exit code {completed.returncode}')
    return completed.stdout.strip()


def installed_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def should_stream_runner_line(line: str) -> bool:
    normalized = line.strip()
    if not normalized:
        return False
    lowered = normalized.lower()
    return (
        normalized.startswith('[')
        or normalized.startswith(f'{EXPECTED_STAGE}:')
        or normalized.startswith('FINAL_REPORT=')
        or any(
            marker in lowered
            for marker in ('traceback', 'error', 'failed', 'failure', 'non-finite')
        )
    )


def expected_exposure_counts(audit: dict) -> dict:
    expected = audit.get('expected', {})
    return {
        'total': expected.get('total'),
        'endpoint': expected.get('shared_endpoint'),
        'stratified': expected.get('stratified'),
    }


def final_comparison_pages(report: dict) -> list[Path]:
    evaluations = report['stages'][EXPECTED_STAGE].get('evaluations', [])
    full = [entry for entry in evaluations if entry.get('kind') == 'full-generation']
    if not full:
        return []
    candidates: list[str] = []

    def collect(value, key_hint: str = '') -> None:
        if isinstance(value, dict):
            for key, child in value.items():
                collect(child, key.lower())
        elif isinstance(value, list):
            for child in value:
                collect(child, key_hint)
        elif isinstance(value, str) and ('sheet' in key_hint or 'page' in key_hint):
            candidates.append(value)

    collect(full[-1])
    pages: list[Path] = []
    seen: set[Path] = set()
    for value in candidates:
        candidate = Path(value)
        if not candidate.is_absolute():
            candidate = REPORT_PATH.parent / candidate
        candidate = candidate.resolve()
        if candidate in seen or not candidate.is_file():
            continue
        seen.add(candidate)
        pages.append(candidate)
    return pages


def schedule_modal_shutdown() -> None:
    if not AUTO_SHUTDOWN:
        print('Automatic shutdown disabled by DIFFUSION_ARC_AUTO_SHUTDOWN=0.')
        return
    modal_task_id = os.environ.get('MODAL_TASK_ID', '').strip()
    if not modal_task_id:
        print('WARNING: MODAL_TASK_ID is unavailable; stop the container manually.')
        return
    shutdown_code = (
        'import subprocess, sys, time; time.sleep(15); '
        "subprocess.run(['modal', 'container', 'stop', '--yes', sys.argv[1]], check=False)"
    )
    subprocess.Popen(
        [sys.executable, '-c', shutdown_code, modal_task_id],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    print('Modal shutdown scheduled in 15 seconds.')


print('Run ID:', RUN_ID)
print('Protocol:', PROTOCOL)
print('Requested source revision:', SOURCE_REVISION or '<missing>')
print('Parent report:', PARENT_REPORT_PATH)
print('Final report:', REPORT_PATH)


## 2. Verify lineage, continue training, evaluate, and shut down

The protected cell verifies the parent report and every parent checkpoint shard, checks out the requested source commit, runs the full unit-test suite, and launches global steps 513–2048. It runs the complete 641-view fixed-noise `t=1` diagnostic at steps 1024 and 2048, but performs costly 50-step generation only at 2048. The step-2048 checkpoint is uploaded before final generation so it survives a later evaluation failure.

In [ ]:
REPORT = None
run_succeeded = False
runner_tail: deque[str] = deque(maxlen=80)

try:
    require_exact_commit(SOURCE_REVISION)
    if not VOLUME_ROOT.is_dir():
        raise FileNotFoundError(f'Modal Volume mount is missing: {VOLUME_ROOT}')
    if not PARENT_REPORT_PATH.is_file():
        raise FileNotFoundError(f'Parent report is missing: {PARENT_REPORT_PATH}')
    parent_sha256 = sha256_file(PARENT_REPORT_PATH)
    if parent_sha256 != PARENT_REPORT_SHA256:
        raise RuntimeError(
            f'Parent report SHA-256 differs: {parent_sha256} versus '
            f'{PARENT_REPORT_SHA256}'
        )
    if REPORT_PATH.exists():
        raise FileExistsError(REPORT_PATH)

    gpu_summary = run_command(
        'query GPU',
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    )
    print('GPU:', gpu_summary)

    SOURCE_DIR.mkdir(parents=True)
    run_command('git init', ['git', 'init'], cwd=SOURCE_DIR)
    run_command(
        'add origin',
        ['git', 'remote', 'add', 'origin', REPOSITORY_URL],
        cwd=SOURCE_DIR,
    )

    github_token = os.environ.get('GITHUB_TOKEN', '').strip()
    if not github_token:
        raise RuntimeError('Attach a Modal Secret containing GITHUB_TOKEN')
    credential = base64.b64encode(
        f'x-access-token:{github_token}'.encode('utf-8')
    ).decode('ascii')
    git_environment = os.environ.copy()
    git_environment.pop('GITHUB_TOKEN', None)
    git_environment.update({
        'GIT_TERMINAL_PROMPT': '0',
        'GIT_CONFIG_COUNT': '1',
        'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
        'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {credential}',
    })
    try:
        run_command(
            'fetch source',
            ['git', 'fetch', '--depth=1', 'origin', SOURCE_REVISION],
            cwd=SOURCE_DIR,
            env=git_environment,
        )
        run_command(
            'checkout source',
            ['git', 'checkout', '--detach', 'FETCH_HEAD'],
            cwd=SOURCE_DIR,
        )
    finally:
        git_environment.clear()
        del credential

    resolved_source_commit = run_command(
        'resolve source',
        ['git', 'rev-parse', 'HEAD'],
        cwd=SOURCE_DIR,
    )
    if resolved_source_commit.lower() != SOURCE_REVISION.lower():
        raise RuntimeError('Fetched source commit differs from requested revision')

    run_command(
        'remove incompatible torchvision',
        [sys.executable, '-m', 'pip', 'uninstall', '--yes', 'torchvision'],
    )
    requirements = [
        f'{name}=={version}'
        for name, version in PINNED_PACKAGES.items()
        if installed_version(name) != version
    ]
    if requirements:
        run_command(
            'install pinned dependencies',
            [
                sys.executable, '-m', 'pip', 'install', '--quiet',
                '--no-input', '--no-cache-dir', *requirements,
            ],
        )
    run_command(
        'install Diffusion-ARC source',
        [
            sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input',
            '--no-deps', '--no-build-isolation', '-e', str(SOURCE_DIR),
        ],
    )

    source_path = str((SOURCE_DIR / 'src').resolve())
    test_environment = os.environ.copy()
    test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
    test_environment['PYTHONPATH'] = source_path
    test_started = time.perf_counter()
    test_process = subprocess.run(
        [sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests'],
        cwd=SOURCE_DIR,
        env=test_environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if test_process.returncode != 0:
        print('\n'.join(test_process.stdout.splitlines()[-50:]))
        raise RuntimeError(f'Unit tests returned {test_process.returncode}')
    print(
        f'Resolved source {resolved_source_commit}; unit tests PASS in '
        f'{time.perf_counter() - test_started:.1f}s.'
    )

    runner = (
        SOURCE_DIR / 'scripts' / 'experiments'
        / 'run_single_task_augmentation_generalization.py'
    )
    if not runner.is_file():
        raise RuntimeError(
            'Selected source revision lacks the augmentation-generalization runner'
        )
    runner_command = [
        sys.executable, '-u', str(runner), '--protocol', PROTOCOL,
        '--volume-root', str(VOLUME_ROOT), '--volume-name', VOLUME_NAME,
        '--run-id', RUN_ID, '--source-commit', resolved_source_commit,
        '--parent-report', str(PARENT_REPORT_PATH),
        '--parent-report-sha256', PARENT_REPORT_SHA256,
    ]
    runner_environment = os.environ.copy()
    runner_environment.update({
        'PYTHONDONTWRITEBYTECODE': '1',
        'PYTHONPATH': source_path,
        'HF_HUB_OFFLINE': '1',
        'TRANSFORMERS_OFFLINE': '1',
        'HF_HUB_DISABLE_TELEMETRY': '1',
        'HF_HUB_DISABLE_PROGRESS_BARS': '1',
        'TOKENIZERS_PARALLELISM': 'false',
    })
    process = subprocess.Popen(
        runner_command,
        cwd=SOURCE_DIR,
        env=runner_environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for raw_line in process.stdout:
        for fragment in raw_line.replace('\r', '\n').splitlines():
            normalized = fragment.strip()
            if not normalized:
                continue
            runner_tail.append(normalized)
            if should_stream_runner_line(normalized):
                print(normalized)
    return_code = process.wait()
    if return_code != 0:
        if REPORT_PATH.is_file():
            failed_report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
            print(json.dumps({
                'status': failed_report.get('status'),
                'failed_stage': failed_report.get('failed_stage'),
                'failures': failed_report.get('failures'),
                'saved_checkpoint': failed_report.get('stages', {}).get(
                    EXPECTED_STAGE, {}
                ).get('saved_checkpoint'),
            }, indent=2))
        raise RuntimeError(
            f'Augmentation-generalization continuation exited with {return_code}'
        )

    REPORT = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
    if REPORT.get('status') != 'COMPLETE':
        raise RuntimeError(f'Experiment report is not COMPLETE: {REPORT.get("status")}')
    if REPORT.get('protocol', {}).get('name') != PROTOCOL:
        raise RuntimeError('Report protocol differs from the requested protocol')

    lineage = REPORT.get('parent_lineage', {})
    expected_lineage = {
        'run_id': PARENT_RUN_ID,
        'report_sha256': PARENT_REPORT_SHA256,
        'optimizer_state_restored': False,
    }
    measured_lineage = {key: lineage.get(key) for key in expected_lineage}
    if measured_lineage != expected_lineage:
        raise RuntimeError(
            f'Parent lineage differs: {measured_lineage} versus {expected_lineage}'
        )
    pool = REPORT.get('pool', {})
    if pool.get('counts') != {'train': 512, 'holdout': 128, 'original': 1}:
        raise RuntimeError('Pool counts differ from the parent protocol')
    if pool.get('matches_parent_exactly') is not True:
        raise RuntimeError('Regenerated pool did not match the parent exactly')

    if set(REPORT.get('stages', {})) != {EXPECTED_STAGE}:
        raise RuntimeError('Report contains an unexpected stage set')
    stage = REPORT['stages'][EXPECTED_STAGE]
    if stage.get('status') != 'COMPLETE':
        raise RuntimeError(f'{EXPECTED_STAGE} is not COMPLETE')
    training = stage.get('training', {})
    expected_training = {
        'completed_steps': 1536,
        'initial_global_step': EXPECTED_INITIAL_STEP,
        'final_global_step': EXPECTED_FINAL_STEP,
        'cumulative_completed_steps': EXPECTED_FINAL_STEP,
        'target_exposures': 12288,
        'cumulative_target_exposures': 16384,
    }
    measured_training = {key: training.get(key) for key in expected_training}
    if measured_training != expected_training:
        raise RuntimeError(
            f'Training budget differs: {measured_training} versus {expected_training}'
        )

    flow_sampling = training.get('flow_sampling', {})
    if flow_sampling.get('update_counts') != {
        'shared_endpoint': 192, 'stratified': 1344,
    }:
        raise RuntimeError('Continuation update counts differ')
    if flow_sampling.get('exposure_counts') != {
        'shared_endpoint': 1536, 'stratified': 10752, 'total': 12288,
    }:
        raise RuntimeError('Continuation exposure counts differ')
    exposure_audit = training.get('per_view_exposure_audit', {})
    if exposure_audit.get('all_exact') is not True:
        raise RuntimeError('Continuation per-view exposure audit failed')
    if expected_exposure_counts(exposure_audit) != {
        'total': 24, 'endpoint': 3, 'stratified': 21,
    }:
        raise RuntimeError('Continuation per-view exposure contract differs')
    cumulative = REPORT.get('training_schedule', {}).get(
        'cumulative_per_view_exposure_audit', {}
    )
    if cumulative.get('all_exact') is not True:
        raise RuntimeError('Cumulative per-view exposure audit failed')
    if expected_exposure_counts(cumulative) != {
        'total': 32, 'endpoint': 4, 'stratified': 28,
    }:
        raise RuntimeError('Cumulative per-view exposure contract differs')
    schedule = REPORT.get('training_schedule', {})
    if schedule.get('global_step_range') != [513, 2048]:
        raise RuntimeError('Continuation global step range differs')
    if schedule.get('parent_flow_seeds_disjoint') is not True:
        raise RuntimeError('Continuation reused parent flow seeds')

    models = REPORT.get('models', {})
    if models.get('warm_start_from_parent_transformer') is not True:
        raise RuntimeError('Transformer was not warm-started from the parent')
    if models.get('optimizer_state_restored') is not False:
        raise RuntimeError('Optimizer reset was not recorded explicitly')

    evaluations = stage.get('evaluations', [])
    evaluation_steps = [entry.get('step') for entry in evaluations]
    evaluation_kinds = [entry.get('kind') for entry in evaluations]
    if evaluation_steps != EXPECTED_EVALUATION_STEPS:
        raise RuntimeError(
            f'Evaluation steps differ: {evaluation_steps} versus '
            f'{EXPECTED_EVALUATION_STEPS}'
        )
    if evaluation_kinds != EXPECTED_EVALUATION_KINDS:
        raise RuntimeError(
            f'Evaluation kinds differ: {evaluation_kinds} versus '
            f'{EXPECTED_EVALUATION_KINDS}'
        )
    for evaluation in evaluations:
        summaries = evaluation.get('t1_diagnostic', {}).get('summaries', {})
        counts = {
            name: summaries.get(name, {}).get('count')
            for name in ('seen', 'holdout', 'original')
        }
        if counts != {'seen': 512, 'holdout': 128, 'original': 1}:
            raise RuntimeError(f't=1 diagnostic counts differ at {evaluation.get("step")}')

    final_panels = evaluations[-1].get('panels', {})
    measured_final_panels = {
        name: final_panels.get(name, {}).get('count')
        for name in ('seen', 'holdout', 'original')
    }
    if measured_final_panels != {'seen': 128, 'holdout': 128, 'original': 1}:
        raise RuntimeError(f'Final panel counts differ: {measured_final_panels}')

    checkpoint = stage.get('saved_checkpoint')
    if not isinstance(checkpoint, dict):
        raise RuntimeError('Final transformer checkpoint is missing')
    if checkpoint.get('step') != EXPECTED_FINAL_STEP:
        raise RuntimeError('Final transformer checkpoint step is not 2048')
    if not checkpoint.get('remote_file_sizes_verified'):
        raise RuntimeError('Final transformer checkpoint is not remotely verified')
    durability = REPORT.get('durability', {})
    if not durability.get('remote_content_verified'):
        raise RuntimeError('Final report content was not verified after upload')

    summary = {
        'run_id': RUN_ID,
        'source_commit': resolved_source_commit,
        'parent_run_id': lineage.get('run_id'),
        'report': str(REPORT_PATH),
        'remote_report': durability.get('remote_report_path'),
        'training': measured_training,
        'evaluation_steps': evaluation_steps,
        'evaluation_kinds': evaluation_kinds,
        'final_panel_counts': measured_final_panels,
        'checkpoint': checkpoint.get('path'),
        'checkpoint_step': checkpoint.get('step'),
    }
    print(json.dumps(summary, indent=2))

    preview_paths = final_comparison_pages(REPORT)
    if not preview_paths:
        print('Final paginated GT-vs-Pred comparison sheets are not present.')
    else:
        try:
            from IPython.display import Image as DisplayImage, display

            print(
                f'Final comparison pages: {len(preview_paths)}; previewing '
                f'{min(len(preview_paths), PREVIEW_PAGE_LIMIT)}.'
            )
            for page_number, preview_path in enumerate(
                preview_paths[:PREVIEW_PAGE_LIMIT], start=1
            ):
                print(f'Page {page_number}: {preview_path}')
                display(DisplayImage(filename=str(preview_path), width=1100))
            if len(preview_paths) > PREVIEW_PAGE_LIMIT:
                print(
                    f'{len(preview_paths) - PREVIEW_PAGE_LIMIT} additional pages '
                    'remain available beside the report.'
                )
        except Exception as preview_error:
            print(f'Could not preview final comparison pages: {preview_error}')
    run_succeeded = True
finally:
    if not run_succeeded and runner_tail:
        print('Runner output tail:')
        print('\n'.join(tuple(runner_tail)[-20:]))
    schedule_modal_shutdown()


## Interpretation guardrails

- The decisive generalization metric remains exact-grid pass@1/pass@2 on the unchanged 128-view holdout.
- Seen improvement without holdout improvement indicates memorization.
- Improvement on both panels supports learning of the shared task rule.
- Flat `t=1` color accuracy near the trivial baseline means more inference or alternative samplers are not the immediate bottleneck.
- Because AdamW state is reset, comparisons must call this a warm-start continuation rather than an exact resume.